# Lakelink Fiber Network Scenario Twin: Blast-Radius Simulation (`nst_blast_radius`)

ML notebook 1 of 3 under `src/ml/` (D-009 budget). Task P3-BLAST-RADIUS-PERSIST. Tag: `telco_project: network_scenario_twin`.

Re-runnable, deterministic version of the SRLG-aware blast-radius engine (`resources/model_design.md` s3). It reads the published twin tables in `cdm_tmforum.network_scenario_twin` and builds 500 failure scenarios (150 node, 300 span, 50 site). It scores them with the champion `srlg_graph_v1` and the counterfactual challenger `path_only_v1`, which ignores SRLGs and is never promoted. It then asserts the s3.6 and s3.7 gates, compares the result with the published `sim_failure_result`, and logs everything to MLflow (`experiments/nst_models`).

* `write_mode = verify` (default): computes and checks the results, and writes nothing to UC.
* `write_mode = overwrite`: also replaces `sim_failure_result`, using INSERT OVERWRITE so table and column comments are kept.

Outputs are simulations, not inventory records (D-003).

In [0]:
import json, hashlib, datetime
from decimal import Decimal, ROUND_HALF_UP
import numpy as np, pandas as pd, mlflow

dbutils.widgets.dropdown("write_mode", "verify", ["verify", "overwrite"])
WRITE_MODE = dbutils.widgets.get("write_mode")
S = "cdm_tmforum.network_scenario_twin"
SEED, N_TOP_SPANS, N_RANDOM_SPANS = 4471, 150, 148
PINNED_SPANS = ["S-4471", "S-4472"]   # SRLG-88 pair, always simulated
EXPERIMENT = "/Users/stephen.hage@databricks.com/network-scenario-twin/experiments/nst_models"
SIM_RUN_ID = "nst-sim-001"

SRC = ["physical_resource", "logical_resource", "service", "service_level_agreement", "customer"]
src_versions = {t: spark.sql(f"DESCRIBE HISTORY {S}.{t} LIMIT 1").first()["version"] for t in SRC}

pr  = spark.table(f"{S}.physical_resource").toPandas()
lr  = spark.table(f"{S}.logical_resource").select("resource_id", "span_id").toPandas()
sv  = spark.table(f"{S}.service").toPandas()
sla = spark.table(f"{S}.service_level_agreement").select("service_id", "credit_pct").toPandas()
cust = spark.table(f"{S}.customer").select("customer_id", "dual_path_flag").toPandas()
assert not sla.service_id.duplicated().any(), "SLA must be 1:1 with service"
print("write_mode:", WRITE_MODE, "| source Delta versions:", src_versions)

In [0]:
# model_design.md s3.2: failure set = failed resource's spans + every span sharing an SRLG (champion only).
# Logical resource fails if its span_id is in the failure set (span_id is constant down the port->wavelength->OCh chain).
# Active services only. Unprotected DOWN if working path hit; protected/dual_path DOWN if both hit, DEGRADED if one.
# sla_exposure_mrr = SUM(mrr * credit_pct / 100) over DOWN services (data_contracts s2.13).
spans, nodes, sites = (pr[pr.type == t] for t in ("span", "node", "site"))
srlg_of = dict(zip(spans.resource_id, spans.srlg))
srlg_members = spans.dropna(subset=["srlg"]).groupby("srlg").resource_id.apply(set).to_dict()
lr_span = dict(zip(lr.resource_id, lr.span_id))

act = sv[sv.status == "active"].merge(sla, on="service_id", how="left").copy()
act["w"] = act.primary_logical_resource_id.map(lr_span)
act["p"] = act.protect_logical_resource_id.map(lambda x: lr_span.get(int(x)) if pd.notna(x) else None)
act["mrr_d"] = act.mrr.map(Decimal)
UNPROT = act.protection_scheme == "unprotected"
Q = Decimal("0.01")

def base_spans(rtype, rid):
    if rtype == "span":
        return {rid}
    if rtype == "node":
        return set(spans[(spans.site_a_id == rid) | (spans.site_z_id == rid)].resource_id)
    kids = set(nodes[nodes.parent_site_id == rid].resource_id)
    return set(spans[spans.site_a_id.isin(kids) | spans.site_z_id.isin(kids)].resource_id)

def simulate(rtype, rid, srlg_aware=True):
    fs = base_spans(rtype, rid)
    if srlg_aware:
        for s in list(fs):
            g = srlg_of.get(s)
            if isinstance(g, str):
                fs |= srlg_members[g]
    wh, ph = act.w.isin(fs), act.p.isin(fs)
    down = (UNPROT & wh) | (~UNPROT & wh & ph)
    deg = ~UNPROT & (wh ^ ph)
    d, g = act[down], act[deg]
    exp = sum((m * Decimal(str(c)) / 100 for m, c in zip(d.mrr_d, d.credit_pct.fillna(0))), Decimal(0))
    return dict(
        srlg_expanded_resource_ids=sorted(fs),
        affected_services=int(len(d)),
        affected_service_ids=sorted(int(x) for x in d.service_id),
        affected_customers=int(d.customer_id.nunique()),
        affected_customer_ids=sorted(int(x) for x in d.customer_id.unique()),
        dual_path_both_failed_customer_ids=sorted(int(x) for x in d[d.protection_scheme == "dual_path"].customer_id.unique()),
        degraded_services=int(len(g)),
        sla_exposure_raw=exp,
        sla_exposure_mrr=exp.quantize(Q, ROUND_HALF_UP),
        degraded_mrr=sum(g.mrr_d, Decimal(0)).quantize(Q, ROUND_HALF_UP),
        _down_mask=down)

In [0]:
# s3.4: all 150 nodes, 300 spans (S-4471, S-4472 + top 150 by riding active MRR + 148 random with seed 4471, sorted), all 50 sites.
riding = pd.concat([act[["w", "mrr_d"]].rename(columns={"w": "s"}),
                    act[["p", "mrr_d"]].rename(columns={"p": "s"})]).dropna()
span_mrr = riding.groupby("s").mrr_d.sum()
cand = pd.DataFrame({"s": sorted(spans.resource_id)})
cand["m"] = cand.s.map(span_mrr).fillna(Decimal(0)).astype(float)
cand = cand[~cand.s.isin(PINNED_SPANS)].sort_values(["m", "s"], ascending=[False, True])
top_spans = list(cand.s[:N_TOP_SPANS])
rest = sorted(set(spans.resource_id) - set(top_spans) - set(PINNED_SPANS))
rand_spans = sorted(np.random.default_rng(SEED).choice(rest, N_RANDOM_SPANS, replace=False).tolist())

scenarios = ([("node", r) for r in sorted(nodes.resource_id)]
             + [("span", r) for r in PINNED_SPANS + top_spans + rand_spans]
             + [("site", r) for r in sorted(sites.resource_id)])
assert len(scenarios) == 500 and len({r for _, r in scenarios}) == 500
print("scenarios:", pd.Series([t for t, _ in scenarios]).value_counts().to_dict())

In [0]:
OUT_COLS = ["srlg_expanded_resource_ids", "affected_services", "affected_service_ids", "affected_customers",
            "affected_customer_ids", "dual_path_both_failed_customer_ids", "degraded_services", "sla_exposure_mrr", "degraded_mrr"]
champ, chall, raw = [], [], {}
for i, (t, r) in enumerate(scenarios, start=1):
    sim_id = f"SIM-{i:04d}"
    o = simulate(t, r, True); c = simulate(t, r, False)
    raw[r] = (o, c)
    champ.append({"sim_id": sim_id, "scenario_type": t, "resource_id": r, **{k: o[k] for k in OUT_COLS}})
    chall.append({"sim_id": sim_id, "scenario_type": t, "resource_id": r, **{k: c[k] for k in OUT_COLS}})

def canon_hash(rows):
    body = json.dumps([{k: (str(v) if isinstance(v, Decimal) else v) for k, v in row.items()} for row in rows], sort_keys=True)
    return hashlib.sha256(body.encode()).hexdigest()
champ_sha = canon_hash(champ)
print("champion rows:", len(champ), "| sha256:", champ_sha)

In [0]:
GT_CUST = [10384, 10897, 12930, 12992, 13642, 14513, 14870, 17014, 17041, 17747, 18793, 19982]
GT_SVC = [19313, 31061, 32479, 37127, 70755, 87059, 94167, 95982, 100034, 100830, 103477, 107506]
dual_flag = sorted(int(x) for x in cust[cust.dual_path_flag == True].customer_id)
o71, c71 = raw["S-4471"]; o72, _ = raw["S-4472"]

gates = {
    "s4471_failure_set": o71["srlg_expanded_resource_ids"] == ["S-4471", "S-4472"],
    "s4471_dual_path_12": o71["dual_path_both_failed_customer_ids"] == GT_CUST,
    "s4471_matches_dual_path_flag": o71["dual_path_both_failed_customer_ids"] == dual_flag,
    "s4471_services": o71["affected_service_ids"] == GT_SVC,
    "s4471_exposure_75472_41": o71["sla_exposure_mrr"] == Decimal("75472.41"),
    "s4471_degraded_0": o71["degraded_services"] == 0,
    "s4472_symmetry": (o72["dual_path_both_failed_customer_ids"] == GT_CUST and o72["sla_exposure_mrr"] == Decimal("75472.41")),
    "path_only_s4471": (c71["affected_services"] == 0 and c71["degraded_services"] == 12
                        and c71["degraded_mrr"] == Decimal("538687.72") and c71["sla_exposure_mrr"] == 0),
    "invariant_cust_le_svc": all(r["affected_customers"] <= r["affected_services"] for r in champ),
    "invariant_exposure_ge_0": all(r["sla_exposure_mrr"] >= 0 for r in champ),
    "row_count_500_unique": len(champ) == 500 and len({r["sim_id"] for r in champ}) == 500,
}

# Property test: 1,000 random sims, no DOWN service may have a surviving path.
rng = np.random.default_rng(SEED)
viol = 0
for idx in rng.choice(len(scenarios), 1000, replace=True):
    t, r = scenarios[idx]
    fs = set(raw[r][0]["srlg_expanded_resource_ids"]); d = act[raw[r][0]["_down_mask"]]
    surv = d.apply(lambda s: (s.w not in fs) if s.protection_scheme == "unprotected" else ((s.w not in fs) or (s.p not in fs)), axis=1)
    viol += int(surv.sum())
gates["property_no_surviving_path"] = viol == 0
gates["deterministic_rerun"] = canon_hash([{"sim_id": f"SIM-{i:04d}", "scenario_type": t, "resource_id": r,
    **{k: simulate(t, r, True)[k] for k in OUT_COLS}} for i, (t, r) in enumerate(scenarios, 1)]) == champ_sha

print(json.dumps(gates, indent=1), "\nS-4471 unrounded exposure:", o71["sla_exposure_raw"])
assert all(gates.values()), f"Gate failures: {[k for k, v in gates.items() if not v]}"

In [0]:
pub = spark.table(f"{S}.sim_failure_result").toPandas().sort_values("sim_id").reset_index(drop=True)
new = pd.DataFrame(champ)
mismatch = {}
if len(pub) == len(new):
    for c in ["sim_id", "scenario_type", "resource_id"] + OUT_COLS:
        # normalize numpy arrays / np.int64 to plain Python so str() comparison is type-neutral
        a = pub[c].map(lambda v: [x.item() if hasattr(x, "item") else x for x in v] if isinstance(v, (list, np.ndarray)) else (v.item() if hasattr(v, "item") and not isinstance(v, Decimal) else v))
        bad = int((a.map(str) != new[c].map(str)).sum())
        if bad: mismatch[c] = bad
else:
    mismatch["row_count"] = (len(pub), len(new))
reproduces_published = not mismatch
print("published rows:", len(pub), "| mismatching columns:", mismatch or "none", "| reproduces published:", reproduces_published)

In [0]:
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name=f"blast_radius_{WRITE_MODE}") as run:
    mlflow.set_tags({"telco_project": "network_scenario_twin", "task": "P3-BLAST-RADIUS-PERSIST", "champion": "srlg_graph_v1",
                     "challenger": "path_only_v1", "simulation_run_id": SIM_RUN_ID, "notebook": "src/ml/nst_blast_radius"})
    mlflow.log_params({"seed": SEED, "n_node": 150, "n_span": 300, "n_site": 50, "status_filter": "active", "write_mode": WRITE_MODE,
                       "exposure_formula": "sum(mrr*credit_pct/100) over DOWN",
                       **{f"src_version_{k}": v for k, v in src_versions.items()}})
    mlflow.log_metrics({**{f"gate_{k}": float(v) for k, v in gates.items()},
        "rows": len(champ), "property_violations": viol, "reproduces_published": float(reproduces_published),
        "s4471_sla_exposure_mrr": float(o71["sla_exposure_mrr"]), "s4471_dual_path_both_failed": len(o71["dual_path_both_failed_customer_ids"]),
        "hidden_exposure_delta_s4471": float(o71["sla_exposure_mrr"] - c71["sla_exposure_mrr"]),
        "total_sla_exposure_mrr_champion": float(sum(r["sla_exposure_mrr"] for r in champ)),
        "total_sla_exposure_mrr_challenger": float(sum(r["sla_exposure_mrr"] for r in chall))})
    mlflow.log_text(champ_sha, "champion_output_sha256_canonical.txt")
    mlflow.log_text(json.dumps(chall, default=str), "challenger_path_only_v1_results.json")
    RUN_ID = run.info.run_id
print("MLflow run:", RUN_ID)

In [0]:
if WRITE_MODE == "overwrite":
    now = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
    out = new.copy()
    for c in ["sla_exposure_mrr", "degraded_mrr"]:
        out[c] = out[c].map(str)
    out = out.assign(model_name="srlg_graph_v1", model_version="v1", mlflow_run_id=RUN_ID, simulated_at=now,
                     origin="SYNTHETIC", simulation_run_id=SIM_RUN_ID)
    spark.createDataFrame(out).createOrReplaceTempView("_nst_sim_failure_new")
    spark.sql(f"""INSERT OVERWRITE {S}.sim_failure_result
        SELECT sim_id, scenario_type, resource_id, srlg_expanded_resource_ids, CAST(affected_services AS INT),
               CAST(affected_service_ids AS ARRAY<BIGINT>), CAST(affected_customers AS INT), CAST(affected_customer_ids AS ARRAY<BIGINT>),
               CAST(dual_path_both_failed_customer_ids AS ARRAY<BIGINT>), CAST(degraded_services AS INT),
               CAST(sla_exposure_mrr AS DECIMAL(18,2)), CAST(degraded_mrr AS DECIMAL(18,2)), model_name, model_version,
               mlflow_run_id, CAST(simulated_at AS TIMESTAMP), origin, simulation_run_id FROM _nst_sim_failure_new""")
    chk = spark.sql(f"SELECT count(*) n, count(DISTINCT sim_id) u, max(CASE WHEN resource_id='S-4471' THEN sla_exposure_mrr END) e FROM {S}.sim_failure_result").first()
    assert chk.n == 500 and chk.u == 500 and chk.e == Decimal("75472.41"), chk
    print("Overwrote sim_failure_result:", chk)
else:
    print("verify mode: no UC writes. Set write_mode=overwrite to republish.")